# Solar Power Plant Analysis — Colab (Drive import version)

This notebook is a Google Colab friendly version of your local `solar_power_plant_analysis.ipynb` focused on mounting Google Drive and loading the Excel files from Drive.
Follow the cells below: mount Drive, set the Drive folder path where you placed the XLSX files, and run the loader to import them into pandas DataFrames.

In [ ]:
# 1) Mount Google Drive (run this in Colab)
from google.colab import drive
drive.mount('/content/drive')

# After this you will be prompted to authorize access to your Drive.

## 2) Point to the Drive folder that contains the Excel files
Set `DRIVE_PATH` to the folder in your Drive where you uploaded the Excel files (for example `'/content/drive/MyDrive/YourFolder'`). If you put the files directly under `MyDrive`, set `DRIVE_PATH = '/content/drive/MyDrive'`.

In [ ]:
# 2a) Set folder Drive tempat file dataset berada
DRIVE_PATH = '/content/drive/MyDrive' 

# Hanya gunakan satu file: cari file yang mengandung kata 'Dataset' dan ber-ekstensi .xlsx
# Jika nama pastinya adalah 'Solar Power Plant Dataset XLSX.xlsx' biarkan saja; kalau berbeda asalkan ada 'Dataset'
DATASET_PATTERN_KEYWORD = 'Dataset'

import os, glob, pandas as pd

def find_dataset_file(drive_path, keyword):
    base = drive_path.rstrip('/')
    pattern = os.path.join(base, '**', f'*{keyword}*.xlsx')
    matches = glob.glob(pattern, recursive=True)
    return matches

# Cari file dataset
found_dataset = find_dataset_file(DRIVE_PATH, DATASET_PATTERN_KEYWORD)
if found_dataset:
    DATASET_PATH = found_dataset[0]
    print('Dataset ditemukan:', DATASET_PATH)
    if len(found_dataset) > 1:
        print(f'Ada {len(found_dataset)} kandidat, gunakan pertama. Jika salah, set DATASET_PATH manual.')
else:
    DATASET_PATH = None
    print('File dataset tidak ditemukan. Pastikan kata "Dataset" ada di nama file dan path benar.')

In [ ]:
# 3) Load dataset Excel menjadi DataFrame tunggal
import pandas as pd, os

if DATASET_PATH is None:
    raise FileNotFoundError('Dataset belum ditemukan. Jalankan sel sebelumnya dan periksa DRIVE_PATH / keyword.')

print('Memuat dataset dari:', DATASET_PATH)
try:
    df = pd.read_excel(DATASET_PATH, engine='openpyxl')
    print('Berhasil memuat dataset. Shape =', df.shape)
except Exception as e:
    raise RuntimeError(f'Gagal membaca file: {e}')

# Preview awal
display(df.head())
print('\nInfo kolom:')
print(df.dtypes)

# Opsional: deteksi kolom tanggal otomatis
possible_datetime = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower() or 'datetime' in c.lower()]
if possible_datetime:
    print('\nKolom kandidat datetime:', possible_datetime)


## 6) Setup Analisis Lanjutan
Menambahkan library, helper functions, dan pipeline analisis sama seperti versi notebook lokal.

In [ ]:
# Import library untuk analisis lanjutan
import glob, math, os, re, warnings
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries (opsional)
try:
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
    print("✓ Scikit-learn loaded successfully")
except ImportError:
    SKLEARN_OK = False
    print("⚠️ Scikit-learn not available (pip install scikit-learn jika perlu)")

# Plotly (opsional)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_OK = True
    print("✓ Plotly loaded successfully")
except ImportError:
    PLOTLY_OK = False
    print("⚠️ Plotly not available - menggunakan matplotlib saja")

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('🔧 Setup completed (Colab)')

In [ ]:
# Helper functions (adaptasi dari notebook asli)
STANDARD_COLS = {
    "date_hour": ["Date - Hour (NMT)", "Date – Hour (NMT)", "Date—Hour (NMT)",
                  "Date Hour (NMT)", "Date_Hour_NMT", "Date/Hour", "Datetime", "DateTime"],
    "wind_speed": ["Wind Speed","Wind_Speed"],
    "sunshine": ["Sunshine"],
    "air_pressure": ["Air Pressure","Air_Pressure"],
    "radiation": ["Radiation"],
    "air_temperature": ["Air Temperature","Air_Temperature"],
    "relative_air_humidity": ["Relative Air Humidity","Relative_Air_Humidity"],
    "system_production": ["System Production","System_Production","Production",
                         "SystemProduction","Actual","y","Y"]
}

def find_column(df, col_key):
    for col in STANDARD_COLS[col_key]:
        if col in df.columns:
            return col
    return None

def clean_numeric(series):
    if series.dtype == 'O':
        s = series.astype(str)
        s = s.str.replace(r"[^\d,\.\-]+", "", regex=True)
        has_both = s.str.contains(",", na=False) & s.str.contains(r"\.", na=False, regex=True)
        s = np.where(
            has_both,
            s.str.replace(r"\.", "", regex=True).str.replace(",", ".", regex=False),
            s.str.replace(",", ".", regex=False)
        )
        series = pd.Series(s)
    return pd.to_numeric(series, errors='coerce')

def parse_datetime(s):
    def parse_one(x):
        if pd.isna(x):
            return pd.NaT
        if isinstance(x, pd.Timestamp):
            return x
        x_str = str(x).strip()
        formats = ["%d/%m/%Y %H:%M","%m/%d/%Y %H:%M","%d.%m.%Y %H:%M","%Y-%m-%d %H:%M:%S","%Y-%m-%d %H:%M","%d-%m-%Y %H:%M"]
        for fmt in formats:
            try:
                return pd.to_datetime(x_str, format=fmt)
            except:
                continue
        try:
            return pd.to_datetime(x_str, dayfirst=True)
        except:
            return pd.to_datetime(x_str, errors='coerce')
    return s.apply(parse_one)

print('✓ Helper functions defined')

In [ ]:
# Fungsi pemrosesan lanjutan (jika ingin menstandarkan kolom dataset yang sudah dibaca)

def enrich_dataset(df):
    # Cari kolom datetime
    date_col = find_column(df, 'date_hour')
    if date_col is None:
        date_like = [c for c in df.columns if re.search(r'(date|time|hour)', c, re.I)]
        date_col = date_like[0] if date_like else None
    if date_col is None:
        raise ValueError('Kolom tanggal/jam tidak ditemukan!')

    # Parse datetime -> 'timestamp'
    df['timestamp'] = parse_datetime(df[date_col])
    before = len(df)
    df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    after = len(df)
    if before != after:
        print(f'⚠️ Removed {before-after} rows invalid timestamp')

    # Clean numeric
    numeric_keys = ["wind_speed","sunshine","air_pressure","radiation","air_temperature","relative_air_humidity","system_production"]
    for key in numeric_keys:
        col = find_column(df, key)
        if col and col in df.columns:
            df[col] = clean_numeric(df[col])

    # Rename ke standar
    rename_map = {}
    for key in STANDARD_COLS:
        col = find_column(df, key)
        if col:
            rename_map[col] = key
    df = df.rename(columns=rename_map)

    # Feature waktu
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['day_of_year'] = df['timestamp'].dt.dayofyear
    df['is_daytime'] = ((df['hour']>=6) & (df['hour']<=18)).astype(int)
    df['is_peak_solar'] = ((df['hour']>=10) & (df['hour']<=15)).astype(int)

    if 'radiation' in df.columns and 'system_production' in df.columns:
        df['solar_efficiency'] = np.where(df['radiation']>0, df['system_production']/df['radiation'], 0)

    print(f'✅ Enriched dataset: {len(df):,} rows, {df.shape[1]} columns')
    print(f"📊 Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df

# Jalankan enrichment pada df yang sudah dimuat
try:
    df = enrich_dataset(df)
except Exception as e:
    print('Enrichment error:', e)

In [ ]:
# 7) Data Overview
if 'system_production' in df.columns:
    print('='*50)
    print('📊 DATASET OVERVIEW')
    print('='*50)
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print()
    missing_data = df.isnull().sum()
    missing_pct = (missing_data/len(df))*100
    missing_info = pd.DataFrame({'Missing Count': missing_data,'Missing %': missing_pct}).round(2)
    print('Missing Data Summary:')
    print(missing_info[missing_info['Missing Count']>0])
    print()
    prod_stats = df['system_production'].describe()
    print('Production Statistics:')
    print(prod_stats)
    non_zero_prod = df[df['system_production']>0]['system_production']
    if len(non_zero_prod)>0:
        print('\nNon-zero production stats:')
        print(f'Count: {len(non_zero_prod):,} ({len(non_zero_prod)/len(df)*100:.1f}%)')
        print(f'Mean: {non_zero_prod.mean():.2f}')
        print(f'Max: {non_zero_prod.max():.2f}')
    print('\n📋 Data Preview:')
    display(df.head())
else:
    print('⚠️ Kolom system_production tidak ditemukan')

In [ ]:
# 8) Analisis Pola Temporal
if 'system_production' in df.columns:
    hourly_pattern = df.groupby('hour')['system_production'].agg(['mean','max','std']).round(2)
    print('📈 HOURLY PRODUCTION PATTERN')
    print('='*40)
    print(hourly_pattern)
    print()
    fig, axes = plt.subplots(2,2, figsize=(15,10))
    axes[0,0].bar(hourly_pattern.index, hourly_pattern['mean'], color='orange', alpha=0.7)
    axes[0,0].set_title('Average Production by Hour', fontweight='bold')
    axes[0,0].set_xlabel('Hour'); axes[0,0].set_ylabel('Production'); axes[0,0].grid(True, alpha=0.3)
    axes[0,1].plot(hourly_pattern.index, hourly_pattern['mean'],'o-', color='red', linewidth=2,label='Mean')
    axes[0,1].fill_between(hourly_pattern.index, hourly_pattern['mean']-hourly_pattern['std'], hourly_pattern['mean']+hourly_pattern['std'], alpha=0.3, color='red', label='±1 Std')
    axes[0,1].set_title('Hourly Pattern with Std', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)
    weekly_pattern = df.groupby('day_of_week')['system_production'].mean()
    day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    axes[1,0].bar(day_names, weekly_pattern.values, color='green', alpha=0.7)
    axes[1,0].set_title('Average Production by Day', fontweight='bold'); axes[1,0].grid(True, alpha=0.3)
    monthly_pattern = df.groupby('month')['system_production'].mean()
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    available_months = monthly_pattern.index
    axes[1,1].bar([month_names[m-1] for m in available_months], monthly_pattern.values, color='blue', alpha=0.7)
    axes[1,1].set_title('Average Production by Month', fontweight='bold'); axes[1,1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    peak_hour = hourly_pattern['mean'].idxmax(); peak_production = hourly_pattern['mean'].max()
    print('🔍 KEY INSIGHTS:')
    print(f'• Peak production hour: {peak_hour}:00 ({peak_production:.2f})')
    print(f"• Production starts around: {hourly_pattern[hourly_pattern['mean']>0].index.min():02d}:00")
    print(f"• Production stops around: {hourly_pattern[hourly_pattern['mean']>0].index.max():02d}:00")
    print(f"• Most consistent hour: {hourly_pattern['std'].idxmin():02d}:00")
    print(f"• Most variable hour: {hourly_pattern['std'].idxmax():02d}:00")
else:
    print('⚠️ Kolom system_production tidak ada')

In [ ]:
# 9) Analisis Korelasi
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['hour','day_of_week','month','day_of_year','is_daytime','is_peak_solar']
analysis_cols = [c for c in numeric_cols if c not in exclude_cols]
if len(analysis_cols) >= 2:
    corr_matrix = df[analysis_cols].corr()
    print('📊 CORRELATION MATRIX')
    print('='*50)
    print(corr_matrix.round(3))
    plt.figure(figsize=(10,8))
    sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0, square=True, fmt='.3f', cbar_kws={'shrink':0.8})
    plt.title('Correlation Matrix', fontweight='bold', pad=20)
    plt.tight_layout(); plt.show()
    if 'system_production' in analysis_cols:
        prod_corr = corr_matrix['system_production'].drop('system_production').sort_values(ascending=False, key=lambda s: s.abs())
        print('🔗 CORRELATION WITH SYSTEM PRODUCTION:')
        for var, val in prod_corr.items():
            strength = 'Strong' if abs(val)>0.7 else 'Moderate' if abs(val)>0.4 else 'Weak'
            direction = 'positive' if val>0 else 'negative'
            print(f'{var:22s}: {val:6.3f} ({strength} {direction})')
        top_vars = prod_corr.abs().head(3).index.tolist()
        if top_vars:
            n_plots = len(top_vars)
            fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots,4))
            if n_plots==1: axes=[axes]
            for i, var in enumerate(top_vars):
                plot_data = df[[var,'system_production']].dropna()
                axes[i].scatter(plot_data[var], plot_data['system_production'], alpha=0.6, s=20, color=f'C{i}')
                axes[i].set_xlabel(var.replace('_',' ').title())
                axes[i].set_ylabel('System Production')
                axes[i].set_title(f'{var} vs Production\n(r={prod_corr[var]:.3f})', fontweight='bold')
                axes[i].grid(True, alpha=0.3)
                if len(plot_data)>1:
                    z = np.polyfit(plot_data[var], plot_data['system_production'], 1)
                    p = np.poly1d(z)
                    axes[i].plot(plot_data[var], p(plot_data[var]), 'r--', alpha=0.8, linewidth=2)
            plt.tight_layout(); plt.show()
    summary_stats = df[analysis_cols].describe().round(2)
    display(summary_stats.T)
else:
    print('⚠️ Tidak cukup kolom numerik untuk korelasi')

In [ ]:
# 10) Forecasting Sederhana
if 'system_production' in df.columns and SKLEARN_OK:
    model_df = df.copy()
    for lag in [1,2,3,6,12,24]:
        model_df[f'prod_lag_{lag}'] = model_df['system_production'].shift(lag)
    if 'radiation' in model_df.columns:
        for lag in [1,2,3]:
            model_df[f'rad_lag_{lag}'] = model_df['radiation'].shift(lag)
    model_df['hour_sin'] = np.sin(2*np.pi*model_df['hour']/24)
    model_df['hour_cos'] = np.cos(2*np.pi*model_df['hour']/24)
    model_df['month_sin'] = np.sin(2*np.pi*model_df['month']/12)
    model_df['month_cos'] = np.cos(2*np.pi*model_df['month']/12)
    feature_cols = [c for c in model_df.columns if 'lag_' in c] + ['hour_sin','hour_cos','month_sin','month_cos','is_daytime','is_peak_solar']
    for var in ['radiation','air_temperature','sunshine','wind_speed']:
        if var in model_df.columns: feature_cols.append(var)
    model_data = model_df.dropna(subset=feature_cols+['system_production'])
    if len(model_data) > 100:
        print('🤖 FORECASTING MODELS COMPARISON')
        train_size = int(len(model_data)*0.8)
        train_data = model_data.iloc[:train_size]; test_data = model_data.iloc[train_size:]
        X_train = train_data[feature_cols]; y_train = train_data['system_production']
        X_test = test_data[feature_cols]; y_test = test_data['system_production']
        hourly_seasonal = train_data.groupby(['hour','month'])['system_production'].mean()
        hourly_fallback = train_data.groupby('hour')['system_production'].mean()
        def get_seasonal_prediction(h,m):
            return hourly_seasonal.loc[(h,m)] if (h,m) in hourly_seasonal.index else hourly_fallback.get(h,0)
        baseline_pred = [get_seasonal_prediction(r.hour, r.month) for r in test_data.itertuples()]
        baseline_mae = mean_absolute_error(y_test, baseline_pred); baseline_r2 = r2_score(y_test, baseline_pred)
        print(f'Baseline MAE: {baseline_mae:.3f} | R²: {baseline_r2:.3f}')
        lr_model = LinearRegression(); lr_model.fit(X_train, y_train); lr_pred = np.maximum(lr_model.predict(X_test),0)
        lr_mae = mean_absolute_error(y_test, lr_pred); lr_r2 = r2_score(y_test, lr_pred)
        print(f'LinearRegression MAE: {lr_mae:.3f} | R²: {lr_r2:.3f}')
        rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
        rf_model.fit(X_train, y_train); rf_pred = np.maximum(rf_model.predict(X_test),0)
        rf_mae = mean_absolute_error(y_test, rf_pred); rf_r2 = r2_score(y_test, rf_pred)
        print(f'RandomForest MAE: {rf_mae:.3f} | R²: {rf_r2:.3f}')
        models_perf = {'Baseline':{'mae':baseline_mae,'r2':baseline_r2,'pred':baseline_pred}, 'Linear':{'mae':lr_mae,'r2':lr_r2,'pred':lr_pred}, 'RF':{'mae':rf_mae,'r2':rf_r2,'pred':rf_pred}}
        best_name = min(models_perf, key=lambda k: models_perf[k]['mae'])
        best_pred = models_perf[best_name]['pred']
        print(f'🏆 Best: {best_name} (MAE={models_perf[best_name]['mae']:.3f}, R²={models_perf[best_name]['r2']:.3f})')
        fig, axes = plt.subplots(1,2, figsize=(15,6))
        last_week = test_data.tail(24*7)
        axes[0].plot(last_week['timestamp'], last_week['system_production'], 'b-', label='Actual')
        axes[0].plot(last_week['timestamp'], best_pred[-len(last_week):], 'r--', label='Pred')
        axes[0].set_title('Last 7 Days'); axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].tick_params(axis='x', rotation=45)
        axes[1].scatter(y_test, best_pred, alpha=0.6, s=20)
        min_val = min(y_test.min(), min(best_pred)); max_val = max(y_test.max(), max(best_pred))
        axes[1].plot([min_val,max_val],[min_val,max_val],'r--'); axes[1].set_title('Actual vs Pred'); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        if best_name=='RF':
            imp_df = pd.DataFrame({'Feature':feature_cols,'Importance':rf_model.feature_importances_}).sort_values('Importance', ascending=False)
            print('🔍 TOP FEATURES:')
            print(imp_df.head(10))
        # Simple 24h forecast
        last_ts = model_data['timestamp'].max(); next_24h = pd.date_range(last_ts + pd.Timedelta(hours=1), periods=24, freq='H')
        forecast = [{'timestamp':ts,'hour':ts.hour,'predicted_production':get_seasonal_prediction(ts.hour, ts.month)} for ts in next_24h]
        forecast_df = pd.DataFrame(forecast)
        print('\n🔮 Next 24h Forecast (Seasonal Baseline):')
        print(forecast_df.head())
    else:
        print('❌ Data kurang untuk modeling (>100 rows diperlukan)')
else:
    if not SKLEARN_OK:
        print('⚠️ scikit-learn tidak tersedia, lewati modeling')
    else:
        print('⚠️ Kolom system_production tidak ditemukan')

In [ ]:
# 11) Business Insights & Rekomendasi
print('🏭 BUSINESS INSIGHTS & RECOMMENDATIONS')
print('='*60)
if 'system_production' in df.columns:
    total_production = df['system_production'].sum()
    avg_daily_production = df.groupby(df['timestamp'].dt.date)['system_production'].sum().mean()
    max_hourly_production = df['system_production'].max()
    print('📊 KEY PERFORMANCE INDICATORS:')
    print(f'• Total Production (periode): {total_production:,.2f}')
    print(f'• Average Daily Production: {avg_daily_production:,.2f}')
    print(f'• Peak Hourly Production: {max_hourly_production:,.2f}')
    if 'radiation' in df.columns and 'solar_efficiency' in df.columns:
        production_hours = df[df['system_production']>0]
        if len(production_hours)>0:
            avg_efficiency = production_hours['solar_efficiency'].mean()
            print(f'• Average Solar Efficiency: {avg_efficiency:.4f}')
    capacity_factor = (total_production / (max_hourly_production * len(df))) * 100 if max_hourly_production>0 else 0
    print(f'• Capacity Factor: {capacity_factor:.2f}%')
    print('\n⚡ OPERATIONAL INSIGHTS:')
    hourly_avg = df.groupby('hour')['system_production'].mean(); peak_hours = hourly_avg.nlargest(3)
    print('• Top 3 Peak Performance Hours:')
    for h, prod in peak_hours.items():
        print(f'  - {h:02d}:00 → {prod:.2f}')
    production_data = df[df['system_production']>0]
    if len(production_data)>0:
        low_threshold = production_data['system_production'].quantile(0.1)
        low_perf_hours = production_data[production_data['system_production']<low_threshold]
        if len(low_perf_hours)>0:
            print(f'• Low Performance Alert: {len(low_perf_hours)} hours < {low_threshold:.2f}')
            common_low = low_perf_hours['hour'].value_counts().head(3)
            for h, cnt in common_low.items():
                print(f'    - {h:02d}:00 ({cnt}x)')
    if 'radiation' in df.columns:
        radiation_corr = df['radiation'].corr(df['system_production'])
        print(f'• Radiation-Production Correlation: {radiation_corr:.3f}')
    if 'air_temperature' in df.columns:
        temp_corr = df['air_temperature'].corr(df['system_production'])
        print(f'• Temperature-Production Correlation: {temp_corr:.3f}')
    if len(df['month'].unique())>1:
        monthly_avg = df.groupby('month')['system_production'].mean()
        worst_month = monthly_avg.idxmin(); best_month = monthly_avg.idxmax()
        month_names = ['', 'Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
        print(f'• Peak Season: {month_names[best_month]} ({monthly_avg[best_month]:.2f})')
        print(f'• Low Season: {month_names[worst_month]} ({monthly_avg[worst_month]:.2f})')
    daily_prod = df.groupby(df['timestamp'].dt.date)['system_production'].sum()
    cv = daily_prod.std()/daily_prod.mean() if daily_prod.mean()>0 else 0
    print(f'• Daily Production Variability (CV): {cv:.3f}')
    zero_pct = (df['system_production']==0).sum()/len(df)*100
    print(f'• Zero production periods: {zero_pct:.1f}%')
    print('\n🔧 RECOMMENDATIONS:')
    print('• Fokus maintenance di low season')
    print('• Monitoring performa inverter saat peak hours')
    print('• Evaluasi panel jika korelasi radiation rendah')
    print('• Jadwalkan pembersihan rutin')
    print('\n💰 BUSINESS VALUE:')
    print('• Forecast untuk scheduling & trading energi')
    print('• Benchmarking performa & optimasi maintenance')
    print('\n📋 DATA QUALITY:')
    missing_pct = (df.isnull().sum()/len(df)*100).round(2)
    critical = missing_pct[missing_pct>10]
    if len(critical)>0:
        print('⚠️ Missing >10%:')
        for c, pct in critical.items():
            print(f'  - {c}: {pct}%')
    else:
        print('✅ Data quality OK (<10% missing)')
else:
    print('⚠️ system_production tidak tersedia')
print('\n📈 ANALYSIS COMPLETE')

## 4) (Opsional) Jika dataset belum ditemukan
Gunakan sel di bawah ini untuk menampilkan semua file `.xlsx` dalam folder Drive agar Anda dapat memastikan nama file mengandung kata `Dataset`. Ubah keyword atau set manual `DATASET_PATH` bila perlu.

In [ ]:
# (Opsional) Daftar semua .xlsx untuk membantu jika file dataset belum ditemukan
import glob, os
pattern = os.path.join(DRIVE_PATH.rstrip('/'), '**', '*.xlsx')
all_xlsx = glob.glob(pattern, recursive=True)
print(f'Ditemukan {len(all_xlsx)} file .xlsx di bawah {DRIVE_PATH}')
for p in all_xlsx[:200]:
    print('-', p)
if len(all_xlsx) > 200:
    print('... (truncated)')

## 5) Notes and next steps
- If a package like `openpyxl` is missing, run `!pip install openpyxl` in a cell.
- Once files are loaded into the `loaded` dict you can continue with the rest of your analysis: copy the transformation and plotting cells from your local notebook and adapt any local-path usage to the DataFrames in `loaded`.
- To use this notebook in Colab: upload it to your Drive (e.g., to `MyDrive/Colab Notebooks`) or open it from GitHub/Colab. After opening, run the cells in order and set `DRIVE_PATH` to the folder containing your Excel files.